In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd

### 手順①ACSの生データの不要部分(推定誤差)を削除し，
### 手順②geoidの11桁を取り出し，整数型に直す

#### データセット名と値の小数点以下桁数の指定

In [21]:
dataset = "B17001"

n = 3

In [3]:
## ACSの生データの不要部分(推定誤差)を削除し，
## geoidの11桁を取り出し，文字列型に直す
a = range(2010, 2024+1)

for year in a:
    df1 = pd.read_csv(rf"E:\USB_DISK\Jupyterlab Raw Data\米国センサス・ACS\未整理\ACS未整理\{dataset}\ACSDT5Y{year}.{dataset}-Data.csv")

    df2 = df1.drop(columns=df1.columns[df1.columns.str.endswith('M')])
    df3 = df2.drop(columns=df2.columns[df2.columns.str.startswith('Unnamed')])

    df3['geoid'] = df3['GEO_ID'].str[-11:]
    df3['geoid'] = df3['geoid'].astype(str)
    
    
    df3.insert(0, "year", year)

    df3.to_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\推定誤差削除済み\{dataset}_{year}_5years.csv", index = False)

In [4]:
# df3.columns

In [5]:
# df3.head()

In [6]:
# # 各行のデータ型をチェック
# for col in df3.columns[:]:
#     print(col)
#     print(df3.iloc[:][col].map(type).value_counts())
#     print()

In [7]:
df3["geoid"].dtype

<StringDtype(storage='python', na_value=nan)>

In [8]:
# df3[df3["geoid"].isna()]

### 手順③ACSの生データを縦結合し，2010-2024連続版を作る

In [9]:
df2010 = pd.read_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\推定誤差削除済み\{dataset}_2010_5years.csv")
df2011 = pd.read_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\推定誤差削除済み\{dataset}_2011_5years.csv")
df2012 = pd.read_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\推定誤差削除済み\{dataset}_2012_5years.csv")
df2013 = pd.read_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\推定誤差削除済み\{dataset}_2013_5years.csv")
df2014 = pd.read_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\推定誤差削除済み\{dataset}_2014_5years.csv")
df2015 = pd.read_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\推定誤差削除済み\{dataset}_2015_5years.csv")
df2016 = pd.read_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\推定誤差削除済み\{dataset}_2016_5years.csv")
df2017 = pd.read_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\推定誤差削除済み\{dataset}_2017_5years.csv")
df2018 = pd.read_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\推定誤差削除済み\{dataset}_2018_5years.csv")
df2019 = pd.read_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\推定誤差削除済み\{dataset}_2019_5years.csv")
df2020 = pd.read_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\推定誤差削除済み\{dataset}_2020_5years.csv")
df2021 = pd.read_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\推定誤差削除済み\{dataset}_2021_5years.csv")
df2022 = pd.read_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\推定誤差削除済み\{dataset}_2022_5years.csv")
df2023 = pd.read_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\推定誤差削除済み\{dataset}_2023_5years.csv")
df2024 = pd.read_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\推定誤差削除済み\{dataset}_2024_5years.csv")

df_list = [
    df2010,
    df2011,
    df2012,
    df2013,
    df2014,
    df2015,
    df2016,
    df2017,
    df2018,
    df2019,
    df2020,
    df2021,
    df2022,
    df2023,
    df2024
]

# 2011年以降の先頭行は除外して結合
acs_series = pd.concat(
    [df if i == 0 else df.iloc[1:]
     for i, df in enumerate(df_list)],
    ignore_index=True
)

acs_series = acs_series.drop(columns = ["GEO_ID", "NAME"])

acs_series.to_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\生データ縦結合済み\ACS_Tract_{dataset}_2010-2024.csv", index = False)

In [10]:
acs_series[acs_series["geoid"].isna()]#.sum()

,year,B17001I_001E,B17001I_002E,B17001I_003E,B17001I_004E,B17001I_005E,B17001I_006E,B17001I_007E,B17001I_008E,B17001I_009E,...,B17001I_051E,B17001I_052E,B17001I_053E,B17001I_054E,B17001I_055E,B17001I_056E,B17001I_057E,B17001I_058E,B17001I_059E,geoid


### 手順④結合した生データファイルと，辞書ファイルを照合させる。
### 手順⑤そのあと，weightとtractpop_shareでウエイト付けした
### 　最終的なクリーンドファイルを完成させる

#### community x tract x year x weightの辞書ファイル読み込み

In [11]:
dictionary = pd.read_csv(r"E:\USB_DISK\JupyterLab Working Directory\ウェイト済み全トラクト辞書ファイル.csv")

In [12]:
# dictionary["geoid"].dtype

In [13]:
# acs_series["geoid"].dtype

In [14]:
dictionary["geoid"] = dictionary["geoid"].astype(str)

In [15]:
dictionary["geoid"].dtype

<StringDtype(storage='python', na_value=nan)>

In [16]:
# dictionary.head()

#### 辞書ファイルとACSの生データを結合
#### (※左側が辞書(tract重複あり)，右側がACS(tract重複なし)
#### 値を数値型に変換

In [17]:
first_row = acs_series.iloc[[0]]
rest_row = acs_series.iloc[1:]

cols = rest_row.columns.tolist()
cols_to_numeric = cols[1:-1]


rest_row[cols_to_numeric] = (
    rest_row[cols_to_numeric]
    .apply(pd.to_numeric, errors="coerce")
)



merged_rest = dictionary.merge(
    rest_row,
    on = ["geoid", "year"],
    how = "left"
)

merged = pd.concat(
    [first_row, merged_rest],
    ignore_index = True
)


In [18]:
# # きちんと2行目以降が数値型になっているかチェック
# for col in merged.columns[:]:
#     print(col)
#     print(merged.iloc[:][col].map(type).value_counts())
#     print()

In [19]:
# merged.columns

In [20]:
# merged.head()

#### 値にウェイト(weight)をかけて，tractごとの
#### Community_areaに分けた実際の値を求める

In [21]:
df5 = merged.copy()

# 2～10列目を対象（例）
cols = df5.columns[1:-10]

# 後ろから5列目("weight")を掛ける列として使用
result = df5.loc[1:, cols].multiply(df5.iloc[1:, -5], axis=0)

##### 整数値に四捨五入する場合はこちらを実行
df5.loc[1:, cols] = np.where(
    (result > 0) & (result < 1),
    1,
    np.floor(result + 0.5)
)

# ##### 四捨五入しない場合はこちらを実行
#  df5.loc[1:, cols] = result

In [22]:
# df5.head(10)

In [23]:
list(df5.isna().sum())

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1]

In [24]:
len(df5)

11951

In [25]:
df5.head()

,year,B17001I_001E,B17001I_002E,B17001I_003E,B17001I_004E,B17001I_005E,B17001I_006E,B17001I_007E,B17001I_008E,B17001I_009E,...,geoid,namelsad,tractpop,tractce,community_area,weight,comm_pop,pop_share,community,commarea_km2
0,2010,Estimate!!Total,Estimate!!Total!!Income in the past 12 months ...,Estimate!!Total!!Income in the past 12 months ...,Estimate!!Total!!Income in the past 12 months ...,Estimate!!Total!!Income in the past 12 months ...,Estimate!!Total!!Income in the past 12 months ...,Estimate!!Total!!Income in the past 12 months ...,Estimate!!Total!!Income in the past 12 months ...,Estimate!!Total!!Income in the past 12 months ...,...,Geography,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010,15,0,0,0,0,0,0,0,0,...,17031842400,Census Tract 8424,3766.0,842400.0,44.0,1.0,35023.0,0.107529,CHATHAM,82.320670
2,2010,3034,525,274,35,44,44,0,0,0,...,17031840300,Census Tract 8403,4085.0,840300.0,59.0,1.0,15626.0,0.261423,MCKINLEY PARK,39.431800
3,2010,37,14,0,0,0,0,0,0,0,...,17031841100,Census Tract 8411,6970.0,841100.0,34.0,1.0,13595.0,0.512688,ARMOUR SQUARE,27.766196
4,2010,4250,1300,546,103,0,17,25,0,75,...,17031841200,Census Tract 8412,4946.0,841200.0,31.0,1.0,35508.0,0.139293,LOWER WEST SIDE,81.550724


### 人口のデータはこれで整理完了
#### csvファイルに保存

In [26]:
df5.to_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\完成版\ACS_Tract_weighted_{dataset}_2010-2024.csv", index = False)

## 所得などの平均値や，割合データの整理
### Community_area内の人口比ウェイトをかけて推定

In [24]:
# ##########################################################################
# ###########手作業で算出した割合値を操作する場合########
# import pandas as pd
# import numpy as np

# rate = pd.read_csv(r"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\完成版\率ACS_Tract_weighted_B17001_2010-2024.csv")

# # 四捨五入で残したい小数点桁数を指定
# n = 5

# # "#DIV/0!" という文字列を欠損値NaNに置換
# rate = rate.replace("#DIV/0!", np.nan)

# col = rate.columns[1:-10]
# print(col)

# # 値を数値型に変更
# rate[col] = rate[col].apply(
#     pd.to_numeric,
#     errors="coerce"
# )

# # 列目を対象
# rate1 = rate.copy()
# cols = rate1.columns[1:-10]

# # 後ろから3列目("pop_share")を掛ける列として使用
# result3 = rate1.loc[:, cols].multiply(rate1.iloc[:, -3], axis=0)

# # #####四捨五入する場合はこれを実行
# rate1.loc[:, cols] = np.floor(result3 * 10**n + 0.5) / 10**n 

# # # ##### 四捨五入しない場合はこちらを実行
# # # rate1.loc[:, cols] = result3

# print(rate1.head(20))

# # csvに保存。（※前のセルの場所と同じ）
# rate1.to_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\完成版\変数ピックアップ済み\率ACS_Tract_weighted_{dataset}_2010-2024.csv", index = False)


##########################################################################
####手順2で作ったファイルを操作するとき##########
# import pandas as pd
# import numpy as np

# # 四捨五入で残したい小数点桁数を指定
# # 列目を対象
# df6 = df5.copy()
# cols = df6.columns[1:-10]

# # 後ろから3列目("pop_share")を掛ける列として使用
# result2 = df6.loc[1:, cols].multiply(df6.iloc[1:, -3], axis=0)

# # #####四捨五入する場合はこれを実行
# df6.loc[1:, cols] = np.floor(result2 * 10**n + 0.5) / 10**n 

# # ##### 四捨五入しない場合はこちらを実行
# # df6.loc[1:, cols] = result2

# # csvに保存。（※前のセルのファイル名，場所と同じ）
# df6.to_csv(rf"E:\USB_DISK\JupyterLab Cleaned File\TractレベルACS\完成版\ACS_Tract_weighted_{dataset}_2010-2024.csv", index = False)